# NHS England A&E — M03 Data Quality Validation

**Unit of analysis:** provider code × reporting month, January 2022–September 2026.

**Input:** M02 combined source-preserving CSV/Parquet (`data/staging/`). The original raw CSVs and staged dataset are **not modified** by M03.

**Goal:** measure completeness, uniqueness, validity, national-total agreement and temporal provider consistency before any cleaning or model development. Run from the cloned `NHS-A-E-IDSS` project in VS Code with its selected Python/Conda kernel.

**Expected M02 inventory (confirmed locally before M03):** 57 source months, 11,449 rows, including 57 national total rows. These are expectations to verify, **not M03 validation findings**.

## 1. Locate the repository and import the validator

The notebook works when VS Code starts the Jupyter kernel from either the repository root or its `notebooks/` folder.

In [ ]:
from pathlib import Path
import sys
import json
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

candidates = [Path.cwd(), *Path.cwd().parents]
repo_root = next((p for p in candidates if (p / "src/data/m03_validate.py").is_file()), None)
if repo_root is None:
    raise FileNotFoundError("Open the NHS-A-E-IDSS folder in VS Code, then run this notebook with its Conda kernel.")
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))
from src.data.m03_validate import validate_dataset
print("Repository:", repo_root)
print("Notebook kernel Python:", sys.executable)

## 2. Execute M03 checks

This step writes grouped issue evidence to `data/quality/` and quality reports to `docs/`. It never changes the M02 input.

In [ ]:
result = validate_dataset(repo_root)
summary = result["summary"]
print(json.dumps(summary, indent=2))

## 3. Review quality findings

**ERROR** = a failed core data-quality check; **WARNING** = a contextual issue to investigate. The M03 quality gate does not substitute for provider reorganisation or research-method review.

In [ ]:
issues = result["findings"].copy()
if issues.empty:
    print("No row-level quality issues were found by the implemented checks.")
else:
    display(issues.sort_values(["severity", "affected_rows"], ascending=[True, False]).head(20))
    print("Unique check types:", issues["check"].nunique())

## 4. Monthly coverage, provider counts and Type 1 zeros

A recorded **zero** Type 1 attendance is not equivalent to a missing value, and can indicate an organisation without Type 1 provision. We do not filter zeros at M03.

In [ ]:
monthly = result["monthly"].sort_values("source_month")
display(monthly.head(6))
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(monthly["source_month"], monthly["provider_rows"], marker=".", linewidth=1.6)
ax.set(title="Reported NHS A&E provider rows by month", xlabel="Reporting month", ylabel="Provider records")
ax.set_xticks(range(0, len(monthly), max(1, len(monthly)//10)))
ax.tick_params(axis="x", labelrotation=40)
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(monthly["source_month"], monthly["type1_zero_provider_share"] * 100, linewidth=1.8)
ax.set(title="Share of provider records reporting zero Type 1 attendances", xlabel="Reporting month", ylabel="Zero Type 1 attendance (%)", ylim=(0, 100))
ax.set_xticks(range(0, len(monthly), max(1, len(monthly)//10)))
ax.tick_params(axis="x", labelrotation=40)
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

## 5. National aggregate reconciliation

For each month and each of the 18 reported count measures, the sum of provider rows is compared with the single national TOTAL row. If any underlying value is malformed or a TOTAL row is missing, that comparison is marked **NOT_COMPARABLE**, not a false match.

In [ ]:
recon = result["reconciliation"]
print("Comparison outcomes:")
display(recon["status"].value_counts().rename_axis("status").reset_index(name="measure_month_checks"))
problems = recon[recon["status"] != "MATCH"]
if not problems.empty:
    display(problems.head(20))
else:
    print("All metric-month comparisons were matched in this run.")

## 6. Organisation continuity and change candidates

Historical NHS provider codes may enter, exit or change names. Internal gaps and large month-on-month Type 1 changes are **review prompts**, not grounds for automatic deletion or imputation.

The change candidate screen uses both `|Δ Type 1| ≥ 2,000` and `|Δ / prior Type 1| ≥ 100%` between consecutive months, purely to prioritise manual investigation.

In [ ]:
continuity = result["continuity"]
print("Providers with internal month gaps:", summary["checks"]["providers_with_internal_month_gaps"])
print("Providers with name variants:", summary["checks"]["providers_with_name_variants"])
display(continuity.sort_values(["missing_within_span", "distinct_provider_names"], ascending=False).head(15))
changes = result["change_candidates"]
print("Large change candidates:", len(changes))
if not changes.empty:
    display(changes.sort_values("absolute_change", key=lambda s: s.abs(), ascending=False).head(15))

## 7. Handover to M04–M05

- **M04 (standardisation):** decide how to represent dates, organisation codes, empty values and published count types, with every transformation documented.
- **M05 (longitudinal integrity):** investigate provider code/name histories, structural zeros and gaps before training forecasting models.
- Do not train a forecasting model from this raw combined table until M03 errors have been investigated and longitudinal modelling scope is defined.

Review the generated report at `docs/m03_quality_report.md` and the detailed CSV evidence before making cleaning decisions.

In [ ]:
print("M03 quality gate:", summary["quality_gate"])
print("Generated files:")
for name, path in result["outputs"].items():
    print(f"  {name:15s} {Path(path).relative_to(repo_root)}")